In [8]:
import cv2
import numpy as np
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk

class ChessboardCornerDetector:
    def __init__(self, root):
        self.root = root
        self.root.title("Rilevatore Incroci Scacchiera")
        
        # Variabili
        self.img_path = ""
        self.corners = []
        self.selected_corners = []
        self.pattern_size = (8, 6)  # Modificare in base alla scacchiera
        
        # Interfaccia
        self.create_widgets()
        
    def create_widgets(self):
        # Frame principale
        self.main_frame = tk.Frame(self.root)
        self.main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Pulsanti
        self.btn_frame = tk.Frame(self.main_frame)
        self.btn_frame.pack(pady=10)
        
        self.load_btn = tk.Button(self.btn_frame, text="Carica Immagine", command=self.load_image)
        self.load_btn.pack(side=tk.LEFT, padx=5)
        
        self.detect_btn = tk.Button(self.btn_frame, text="Rileva Incroci", command=self.detect_corners, state=tk.DISABLED)
        self.detect_btn.pack(side=tk.LEFT, padx=5)
        
        self.reset_btn = tk.Button(self.btn_frame, text="Reset", command=self.reset, state=tk.DISABLED)
        self.reset_btn.pack(side=tk.LEFT, padx=5)
        
        # Canvas per l'immagine
        self.canvas = tk.Canvas(self.main_frame, bg='gray')
        self.canvas.pack(fill=tk.BOTH, expand=True)
        self.canvas.bind("<Button-1>", self.on_canvas_click)
        
        # Etichette informazioni
        self.info_label = tk.Label(self.main_frame, text="Carica un'immagine di una scacchiera", fg="blue")
        self.info_label.pack(pady=5)
        
        self.coord_label = tk.Label(self.main_frame, text="", fg="green")
        self.coord_label.pack(pady=5)
    
    def load_image(self):
        self.img_path = filedialog.askopenfilename(
            title="Seleziona immagine scacchiera",
            filetypes=[("Image files", "*.jpg *.jpeg *.png *.bmp")]
        )
        
        if not self.img_path:
            return
            
        self.original_img = cv2.imread(self.img_path)
        if self.original_img is None:
            tk.messagebox.showerror("Errore", "Impossibile caricare l'immagine")
            return
            
        self.display_image(self.original_img)
        self.detect_btn.config(state=tk.NORMAL)
        self.info_label.config(text="Clicca 'Rileva Incroci' per trovare gli angoli della scacchiera")
    
    def detect_corners(self):
        # Converti in scala di grigi
        gray = cv2.cvtColor(self.original_img, cv2.COLOR_BGR2GRAY)
        
        # Trova gli angoli della scacchiera
        ret, corners = cv2.findChessboardCorners(gray, self.pattern_size, None)
        
        if not ret:
            tk.messagebox.showwarning("Attenzione", "Nessuna scacchiera rilevata!")
            return
            
        # Raffina la posizione degli angoli
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
        corners = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        
        self.corners = [c[0] for c in corners]  # Estrai coordinate
        
        # Mostra gli angoli rilevati
        display_img = self.original_img.copy()
        for i, (x, y) in enumerate(self.corners):
            cv2.circle(display_img, (int(x), int(y)), 8, (0, 255, 0), 2)
            cv2.putText(display_img, str(i+1), (int(x)+10, int(y)-10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
        
        self.display_image(display_img)
        self.reset_btn.config(state=tk.NORMAL)
        self.info_label.config(text=f"Trovati {len(self.corners)} incroci. Seleziona 4 punti cliccando sui cerchi verdi")
    
    def on_canvas_click(self, event):
        if not self.corners or len(self.selected_corners) >= 4:
            return
            
        # Trova l'angolo più vicino al click
        click_pos = np.array([event.x, event.y])
        distances = [np.linalg.norm(np.array(c) - click_pos) for c in self.corners]
        closest_idx = np.argmin(distances)
        
        if distances[closest_idx] < 15:  # Soglia di selezione
            selected_corner = self.corners[closest_idx]
            if closest_idx not in [idx for idx, _ in self.selected_corners]:
                self.selected_corners.append((closest_idx, selected_corner))
                
                # Disegna il marker di selezione
                display_img = self.original_img.copy()
                for i, (x, y) in enumerate(self.corners):
                    color = (0, 255, 0)  # Verde per angoli non selezionati
                    if i in [idx for idx, _ in self.selected_corners]:
                        color = (0, 0, 255)  # Rosso per angoli selezionati
                    cv2.circle(display_img, (int(x), int(y)), 8, color, 2)
                
                self.display_image(display_img)
                
                if len(self.selected_corners) == 4:
                    self.show_selected_coordinates()
    
    def show_selected_coordinates(self):
        coord_text = "Coordinate selezionate:\n"
        for i, (idx, (x, y)) in enumerate(self.selected_corners, 1):
            coord_text += f"Punto {i}: Angolo {idx+1} - X={x:.1f} px, Y={y:.1f} px\n"
        
        self.coord_label.config(text=coord_text)
        self.info_label.config(text="Selezione completata! Premi Reset per ricominciare")
    
    def display_image(self, img):
        # Converti da BGR a RGB
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_pil = Image.fromarray(img_rgb)
        
        # Ridimensiona se necessario
        max_size = (800, 600)
        if img_pil.width > max_size[0] or img_pil.height > max_size[1]:
            img_pil.thumbnail(max_size, Image.LANCZOS)
        
        self.tk_img = ImageTk.PhotoImage(img_pil)
        
        # Aggiorna canvas
        self.canvas.delete("all")
        self.canvas.config(width=self.tk_img.width(), height=self.tk_img.height())
        self.canvas.create_image(0, 0, anchor=tk.NW, image=self.tk_img)
    
    def reset(self):
        self.corners = []
        self.selected_corners = []
        self.display_image(self.original_img)
        self.coord_label.config(text="")
        self.info_label.config(text="Clicca 'Rileva Incroci' per trovare gli angoli della scacchiera")

if __name__ == "__main__":
    root = tk.Tk()
    app = ChessboardCornerDetector(root)
    root.mainloop()